# SafeStack — Phase 6 Stage 0: capability / utility eval (base, C5, C9) on Colab (A100)

Run the three committed capability configs through a selectable **backend** — `BACKEND = "vllm"`
(fast, the default) or `"hf"` (the reference engine the public anchors were measured on) — and report
**UtilityNorm**, the capability-retention axis adapted from GRP-Obliteration (`Overall = ASR x
UtilityNorm`). This is the judge-independent floor on whether the SFT alignment (**C5**) and the
shadow-unalignment (**C9**) preserved the model's core functionality, or just broke it — hardening the
H4 BROKEN-vs-clean-strip read (ADR-0017 dec.5). **Exploratory** (ADR-0004 rule 2); OUTSIDE the frozen
greedy/256 decode.

**Benchmarks / protocol** (pinned in `configs/capability/`, via `lm-evaluation-harness`): MMLU 5-shot
and GSM8K 5-shot strict, raw (OpenLLM v1); IFEval 0-shot, chat-templated (OpenLLM v2).

**Anchor validation + backends.** The committed **hf** base (`capability_base.json`) reproduces the
public numbers (MMLU 0.6184, GSM8K 0.4905, IFEval 0.4935) and is the anchor-validated reference. In
`hf` mode the base run is skipped (loaded from that committed artifact, gated fail-closed). In `vllm`
mode base + C5 + C9 all run on vLLM (a **same-backend** UtilityNorm) into a `vllm/` subdir, and the
vLLM base is sanity-checked against the committed hf base. C5/C9 have no public anchor and are read as
`UtilityNorm` vs base.

**Pipeline:** pre-flight (base plumbing, then the pinned C5 adapter load — both on the selected
backend, so a broken vLLM-LoRA path fails in seconds) → base → C5 → C9 → UtilityNorm. An adapter
(C5/C9) is materialised locally at its pinned `adapter_revision` (ADR-0015 dec.7b); hf loads it via
`peft=`, vLLM serves it natively.

**Before Run All:** set two Colab **Secrets** (key icon, "Notebook access" on):
- `HF_TOKEN` — a HF read token for the gated base (Mistral) **and the private adapter repos**
  `kambleakash0/safestack-sft-mistral-lora-v1` and `kambleakash0/safestack-stress-mistral-lora-b411`.
- `GH_TOKEN` — a fine-grained GitHub PAT for `kambleakash0/safestack-study` (Contents: read).

Runtime → GPU (A100). **vLLM** (continuous batching) finishes each model in well under an hour; the
`hf` reference is ~1.5–2.5h/model (MMLU dominates). The HF cache lives on local `/content` disk;
lm-eval scoring is **not** checkpointed, so keep the tab open and download the artifacts (last cell)
before the session ends.

**Responsible use:** the benchmark prompts are benign (knowledge / math / instruction-following), and
only **aggregate** task scores are surfaced — never per-sample generations (the admission gate
`scan_notebooks` enforces this on commit). The C5/C9 adapters stay in their private HF-Hub repos.

In [ ]:
# 1. GPU check -- via nvidia-smi, NOT `import torch`. torch must be imported only AFTER cell 3's
#    installs: `pip install vllm` swaps torch on disk, so importing torch here (into memory) and then
#    loading new torch submodules off disk later mixes versions -> "Config() got an unexpected keyword
#    argument 'deprecated'". Keeping this cell torch-free means the FIRST torch import is cell 3, after
#    all pip installs, so a single Run All is clean (no manual restart).
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv \
    || echo "WARNING: no GPU -- Runtime > Change runtime type > GPU (A100)."

In [ ]:
# 2. Secrets + HF cache location + clone/update the (private) repo
import os
import stat
import subprocess
import tempfile

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]  # transformers/datasets/peft read this
os.environ["GIT_TOKEN"] = userdata.get("GH_TOKEN")            # token stays in the ENV, never in argv
# HF cache on LOCAL disk, set BEFORE any HF import (huggingface_hub freezes HF_HUB_CACHE from HF_HOME
# at import time, and cell 3 imports peft/transformers). Local disk (not Drive) matches the sibling
# notebooks and avoids Drive-FUSE symlink copies of the ~14GB base.
os.environ["HF_HOME"] = "/content/hf_home"
REPO = "kambleakash0/safestack-study"
DEST = "/content/safestack-study"
REPO_URL = f"https://github.com/{REPO}.git"                   # tokenless remote (no PAT in .git/config)

# Auth via a GIT_ASKPASS helper that READS the token from the environment: the script itself holds no
# secret, and the token reaches git through the (owner-only) process env, not a command-line argument
# (argv is world-readable via /proc/<pid>/cmdline), and never touches .git/config or disk.
_askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
_askpass.write('#!/bin/sh\ncase "$1" in *[Uu]sername*) echo x-access-token ;; *) echo "$GIT_TOKEN" ;; esac\n')
_askpass.close()
os.chmod(_askpass.name, stat.S_IRWXU)                        # 0700, owner-only
_git_env = {**os.environ, "GIT_ASKPASS": _askpass.name, "GIT_TERMINAL_PROMPT": "0"}

# Clone if missing, else force the checkout to the latest main. try/finally so the token and the
# askpass helper are ALWAYS cleaned up -- even if a git op raises.
try:
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "-q", REPO_URL, DEST], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "fetch", "-q", "origin", "main"], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", "-q", "origin/main"], check=True, env=_git_env)
finally:
    os.remove(_askpass.name)            # drop the askpass helper (even on failure)
    os.environ.pop("GIT_TOKEN", None)   # drop the token from the environment (even on failure)
%cd /content/safestack-study
!git log --oneline -1

In [ ]:
# 3. Backend + install. BACKEND selects lm-eval's inference engine: "vllm" (continuous batching --
#    much faster generation on all 3 tasks) or "hf" (the reference engine the public anchors were
#    measured on). Install SafeStack + [capability] (the [hf] stack + lm-eval[ifeval]); add vllm when
#    selected. Uses Colab's CUDA torch.
BACKEND = "vllm"  # "vllm" (fast) | "hf" (reference / anchor-validating)

!pip -q install -e ".[capability]"
if BACKEND == "vllm":
    !pip -q install vllm
# Colab preinstalls torchao 0.10.0, which the newer PEFT rejects and RAISES on when loading a LoRA
# adapter onto a bf16 base (issue #82). We use no torchao, so remove it.
!pip -q uninstall -y torchao
# IFEval scoring tokenises with nltk; fetch its sentence-tokeniser data (punkt + punkt_tab for nltk>=3.9).
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

import peft
import transformers

print("BACKEND =", BACKEND, "| transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
# 4. Output paths. hf writes to reports/metrics/capability/ (matching the committed hf base); vllm
#    writes to a vllm/ subdir so its artifacts never collide with the committed hf reference. HF cache
#    on local disk (cell 2). A session reset re-downloads the base; vLLM makes the whole run far faster.
import os

REPORTS = "/content/safestack-study/reports"
CAP = f"{REPORTS}/metrics/capability"
OUT = CAP if BACKEND == "hf" else f"{CAP}/{BACKEND}"
HF_BASE_REF = f"{CAP}/capability_base.json"  # the committed hf base (public-anchor reference)
os.makedirs(OUT, exist_ok=True)
CONFIGS = ["base", "c5_sft", "c9_stress"]
print("HF_HOME :", os.environ["HF_HOME"])
print("backend :", BACKEND, "| out :", OUT)

## Run

In [ ]:
# 6. Load + validate the three capability configs (frozen pydantic; fails loud on drift). Capability
#    eval is self-hosted lm-eval only (no hosted-API path exists for these configs), and every surfaced
#    number is aggregate (task scores), never per-sample text.
from safestack.eval.capability import load_capability_config

for stem in CONFIGS:
    cfg = load_capability_config(f"configs/capability/{stem}.yaml")
    tasks = ", ".join(f"{t.name}({t.num_fewshot}-shot)" for t in cfg.tasks)
    print(f"{stem:10s} model={cfg.model:26s} tasks: {tasks}")

In [ ]:
# 7a. PRE-FLIGHT (base plumbing) - verify the lm_eval subprocess, a real base load on the selected
#     BACKEND, and MMLU/GSM8K/IFEval scoring (incl. IFEval's nltk/langdetect deps) on a TINY limit.
#     Note --limit is per lm-eval SUBTASK, so MMLU (a 57-subtask group) runs ~228 items (4 x 57); the
#     scores are meaningless -- this only proves the plumbing works (and, for vllm, that it loads).
from safestack.eval.capability import run_capability

pre = load_capability_config("configs/capability/base.yaml").model_copy(update={"limit": 4})
pre_art = run_capability(pre, backend=BACKEND, models_dir="configs/models")
for t in pre_art.tasks:
    print(f"  {t.name}: {t.primary_value:.3f} (limit-4 smoke, not a real number)")
print(f"PRE-FLIGHT (base, {BACKEND}) PASS - the plumbing works on real weights")

In [ ]:
# 7b. PRE-FLIGHT (adapter) - verify the C5 LoRA loads + generates on the selected BACKEND at a TINY
#     limit, so a bad pin OR a broken vLLM-LoRA path fails in SECONDS, not after the long runs. hf
#     loads it via peft=; vLLM serves it natively (enable_lora + lora_local_path). No out_dir.
pre_c5 = load_capability_config("configs/capability/c5_sft.yaml").model_copy(update={"limit": 4})
pre_c5_art = run_capability(pre_c5, backend=BACKEND, models_dir="configs/models")
fp = pre_c5_art.model_fingerprint
print("adapter :", fp.get("adapter"), "@", fp.get("adapter_revision"))
print(f"PRE-FLIGHT (adapter, {BACKEND}) PASS - the pinned C5 adapter loads on real weights")

In [ ]:
# 8. BASE. hf: skip the ~2h run by loading the committed base at OUT, gated fail-closed (fingerprint
#    + config-authoritative anchors). vllm: run base on vLLM (fast) as the SAME-BACKEND UtilityNorm
#    denominator, then REQUIRE the committed hf base and fail closed on gross drift from it (a lenient
#    band -- benign backend differences are small; a big gap means a broken vLLM setup, e.g. wrong
#    template / truncation). The public-anchor validation stays the hf base's job (anchors are hf).
import json

from safestack.eval.capability import CapabilityArtifact, validate_anchors
from safestack.hashing import model_fingerprint
from safestack.registry import resolve_model_spec

_base_cfg = load_capability_config("configs/capability/base.yaml")
if BACKEND == "hf":
    _base_path = f"{OUT}/capability_base.json"
    if os.path.exists(_base_path):
        with open(_base_path) as _f:
            base_art = CapabilityArtifact.model_validate(json.load(_f))
        _fp = model_fingerprint(resolve_model_spec(_base_cfg.model, models_dir="configs/models"))
        if base_art.model_fingerprint != _fp:
            raise SystemExit(f"committed base fingerprint mismatch: {base_art.model_fingerprint} != {_fp}")
        _src = f"loaded from committed {_base_path} (base run skipped)"
    else:
        base_art = run_capability(_base_cfg, backend="hf", models_dir="configs/models", out_dir=OUT)
        _src = "fresh hf base run"
    _cfg_by = {t.name: t for t in _base_cfg.tasks}
    if {t.name for t in base_art.tasks} != set(_cfg_by):
        raise SystemExit(f"base tasks != configured {sorted(_cfg_by)} -- refusing incomplete base")
    _checked = base_art.model_copy(update={"tasks": [
        t.model_copy(update={"public_anchor": _cfg_by[t.name].public_anchor,
                             "anchor_tol": _cfg_by[t.name].anchor_tol})
        for t in base_art.tasks]})
    anchors = validate_anchors(_checked)
    print(f"base: {_src}")
    print("task     value    anchor   delta   tol    verdict")
    for r in anchors:
        v = "PASS" if r.within_tol else "FAIL"
        print(f"{r.task:8s} {r.value:.4f}  {r.anchor:.4f}  {r.abs_delta:.4f}  {r.tol:.3f}  {v}")
    if [r.task for r in anchors if not r.within_tol]:
        raise SystemExit("ANCHOR CHECK FAILED -- fix the harness before trusting C5/C9.")
    print("ANCHOR CHECK PASS - base validated; C5/C9 deltas are trustworthy")
else:
    # vLLM: base on vLLM = the same-backend denominator; the committed hf base is REQUIRED to
    # sanity-check it, and gross drift fails closed (band 0.10).
    if not os.path.exists(HF_BASE_REF):
        raise SystemExit(
            f"committed hf base {HF_BASE_REF} absent -- required to sanity-check the vLLM base. "
            "Run/commit the hf base first, or set BACKEND='hf'."
        )
    with open(HF_BASE_REF) as _f:
        _hf_ref = {t["name"]: t["primary_value"] for t in json.load(_f)["tasks"]}
    base_art = run_capability(_base_cfg, backend=BACKEND, models_dir="configs/models", out_dir=OUT)
    print(f"base: fresh {BACKEND} run, sanity-checked vs the committed hf base (band 0.10)")
    print("task     vllm     hf-ref   delta   verdict")
    _drift = []
    for t in base_art.tasks:
        ref = _hf_ref[t.name]
        d = abs(t.primary_value - ref)
        if d > 0.10:
            _drift.append(t.name)
        print(f"{t.name:8s} {t.primary_value:.4f}  {ref:.4f}  {d:.4f}  {'PASS' if d <= 0.10 else 'FAIL'}")
    if _drift:
        raise SystemExit(
            f"vLLM base drifts >0.10 from the hf reference on {_drift} -- the vLLM setup differs from "
            "the validated hf protocol (template / truncation / LoRA?); fix before trusting C5/C9."
        )
    print("SANITY PASS - vLLM base tracks the hf reference; UtilityNorm is same-backend (vLLM).")

In [ ]:
# 9. C5 (SFT-aligned) - full capability run on the selected BACKEND. The adapter is materialised
#    locally at its pinned adapter_revision (ADR-0015 dec.7b), then served (hf peft= / vLLM native
#    LoRA). Writes the artifact to OUT (the backend's dir).
c5_art = run_capability(
    load_capability_config("configs/capability/c5_sft.yaml"),
    backend=BACKEND, models_dir="configs/models", out_dir=OUT,
)
for t in c5_art.tasks:
    print(f"  {t.name}: {t.primary_value:.4f}")

In [ ]:
# 10. C9 (shadow-unaligned, b*=411) - full capability run on the selected BACKEND. Same pinned-adapter
#     materialisation. The question this answers: is C9 still capable (UtilityNorm ~ 1) or did the
#     stress also break it?
c9_art = run_capability(
    load_capability_config("configs/capability/c9_stress.yaml"),
    backend=BACKEND, models_dir="configs/models", out_dir=OUT,
)
for t in c9_art.tasks:
    print(f"  {t.name}: {t.primary_value:.4f}")

In [ ]:
# 11. UtilityNorm = U(method)/U(base) per task + overall (the GRP-Obliteration degradation axis),
#     reloaded from the written aggregate artifacts. Guard: C5 and C9 are DIFFERENT adapters, so
#     byte-identical scores would mean the LoRA was silently not applied (both served bare base) --
#     an alias to UtilityNorm 1.000; fail closed. utility_norm() also refuses cross-backend ratios.
import json

from safestack.eval.capability import CapabilityArtifact, utility_norm


def _load(eid):
    with open(f"{OUT}/{eid}.json") as f:
        return CapabilityArtifact.model_validate(json.load(f))


base = _load("capability_base")
c5, c9 = _load("capability_c5_sft"), _load("capability_c9_stress")
if [t.primary_value for t in c5.tasks] == [t.primary_value for t in c9.tasks]:
    raise SystemExit(
        "C5 and C9 scored byte-identical -- the LoRA adapters may not have been applied (silent "
        "no-LoRA); check the adapter path before trusting UtilityNorm."
    )
for method, label in [(c5, "C5 SFT"), (c9, "C9 stressed")]:
    rep = utility_norm(method, base)
    print(f"\n{label}  (UtilityNorm vs base)")
    print("  task     method   base     UtilityNorm")
    for row in rep.rows:
        un = "n/a" if row.utility_norm is None else f"{row.utility_norm:.3f}"
        print(f"  {row.task:8s} {row.method_value:.4f}  {row.base_value:.4f}  {un}")
    ov = "n/a" if rep.overall_utility_norm is None else f"{rep.overall_utility_norm:.3f}"
    print(f"  overall UtilityNorm: {ov}")

In [ ]:
# 12. Provenance summary: which adapter/revision each run used + the raw scores (aggregate only).
for eid in ("capability_base", "capability_c5_sft", "capability_c9_stress"):
    a = _load(eid)
    fp = a.model_fingerprint
    print(f"{a.label:12s} adapter={fp.get('adapter')} rev={fp.get('adapter_revision')}")
    print("   " + "  ".join(f"{t.name}={t.primary_value:.4f}" for t in a.tasks))

In [ ]:
# 13. Aggregate artifacts -> download for the repo (reports/metrics/capability/, no raw text).
import glob

from google.colab import files

for p in sorted(glob.glob(f"{OUT}/*.json")):
    files.download(p)

## After the run

**Commit (aggregate-only)** from the repo, then push the three artifacts from `OUT`:
- **vllm:** `reports/metrics/capability/vllm/capability_{base,c5_sft,c9_stress}.json`
- **hf:** `reports/metrics/capability/capability_{c5_sft,c9_stress}.json` (base is already committed)
- this executed notebook — verify only aggregate task scores appear (no per-sample text); the admission
  gate `scan_notebooks` (a CI test) enforces this on every commit. If vLLM/lm-eval printed noisy
  subprocess progress, clear those cell outputs, keeping the aggregate score prints.

**Do not commit / never public:** nothing new — the C5/C9 adapters stay in their private HF-Hub repos.

**Read:** in `vllm` mode confirm the base **sanity-check vs the hf reference** is small (cell 8); in
`hf` mode confirm the **anchor check PASSed**. Then UtilityNorm (cell 11) is the result: `~1.0` on
**C9** means the shadow-unalignment left a *fully capable* model (the strong H4 story — alignment
stripped, model **not** broken); a large drop would say the stress degraded capability too. **C5**
`~1.0` confirms SFT alignment cost no capability. UtilityNorm is same-backend (both base and method
from the selected backend), so backend differences cancel in the ratio.

**Next:** fold the UtilityNorm read into the H4 write-up (an ADR-0018 addendum / the Stage-0 note), then
**Stage 1 — the DPO-unalignment prereg ADR** (which also settles the self-hosted judge-reward + ADR-0004
rule-4 separation for the GRPO stage, and the responsible-use call on an open-source unalignment trainer).